# AR Rule Consistency GPT Baseline

This baseline sends the complete AR image and its application rule directly to `gpt-5.6-luna` with medium reasoning and no visual tools. It evaluates the Agent's detected inconsistency cases using case-level TP, FP, and FN counts.

The run resumes from checkpoint `x` and processes at most `n_samples` new images. Results are saved after every successful image.

- `baseline_core_results.json`: file name, binary match, inconsistency-case TP/FP/FN counts, empty tool counts, and latency.
- `baseline_raw_results.json`: file name and the baseline Agent output only.

No crops, masks, or tool traces are saved.

In [1]:
import base64
import json
import sys
import time
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

workspace_root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "rule_agent_github").is_dir()
)
experiment_dir = workspace_root / "rule_agent_github"
load_dotenv(experiment_dir / ".env")
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

from rule_agent_github.data import find_image, load_metadata

# x is the checkpoint count; n_samples is the limit for this run.
x = 0
n_samples = 108
if not 0 <= x <= 108:
    raise ValueError("x must be between 0 and 108")
if not 0 <= n_samples <= 108 - x:
    raise ValueError("n_samples must be between 0 and 108 - x")

MODEL = "gpt-5.6-sol"
THINKING = "medium"
ANALYSIS_MODEL = "gpt-5.6-luna"
CORE_RESULTS_PATH = experiment_dir / "sol_thinkingmedium_baseline_core_results.json"
RAW_RESULTS_PATH = experiment_dir / "sol_thinkingmedium_baseline_raw_results.json"
client = OpenAI()

print("Baseline model:", MODEL)
print("Reasoning effort:", THINKING)
print("Checkpoint x:", x)
print("Samples this run:", n_samples)

Baseline model: gpt-5.6-sol
Reasoning effort: medium
Checkpoint x: 0
Samples this run: 108


In [2]:
def image_data_url(image_path):
    media_type = {".jpg": "image/jpeg", ".jpeg": "image/jpeg", ".png": "image/png"}.get(image_path.suffix.lower())
    if media_type is None:
        raise ValueError(f"Unsupported image format: {image_path.suffix}")
    encoded = base64.b64encode(image_path.read_bytes()).decode("utf-8")
    return f"data:{media_type};base64,{encoded}"


def load_results(path):
    if not path.exists():
        return []
    with path.open(encoding="utf-8") as result_file:
        value = json.load(result_file)
    if not isinstance(value, list):
        raise ValueError(f"Result file must contain a JSON list: {path}")
    return value


def save_results(path, records):
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    temporary_path.write_text(json.dumps(records, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    temporary_path.replace(path)


def analyze_baseline_output(record, agent_output):
    prompt = f"""
Compare the baseline GPT output with the reference annotation. Return one valid
JSON object only with exactly these keys: prediction, inconsistency_tp,
inconsistency_fp, inconsistency_fn, and report. Prediction must be exactly
CONSISTENT or INCONSISTENT. The three inconsistency values must be non-negative
integers.

The TP/FP/FN are case-level counts for inconsistency prediction inside this
image, not image-level counts and not generic description-quality scores. A
case means one distinct rule-violation event reported by the Agent.

First extract the distinct violation cases from the Agent output and the
reference descriptive annotation. For a Consistent reference image, there are
zero reference violation cases: every violation case reported by the Agent is
FP, with TP=0 and FN=0. For Omission, Commission, or Confusion images, match
Agent-reported violation cases to reference violation cases by semantic
compatibility. A broader description or a different name is acceptable when it
clearly refers to the same violation case. Matched cases are TP, Agent-only
cases are FP, and reference-only cases are FN. If a case is not reported, it is
not a TP. Do not collapse multiple distinct violation cases into one just
because they share the same inconsistency type.

The prediction field is the Agent's overall binary conclusion, while the
three counts measure the individual inconsistency cases behind that conclusion.

Reference inconsistency type: {record["Inconsistency Type"]}
Reference binary annotation: {record["Binary Annotation"]}
Reference descriptive annotation: {record["Descriptive Annotation"]}
Agent output:
{agent_output}

The report must briefly state the overall binary comparison and how individual
violation cases were matched and counted. Do not invent tool calls; this
baseline uses no tools.
"""
    response = client.responses.create(model=ANALYSIS_MODEL, input=prompt)
    text = response.output_text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    analysis = json.loads(text)
    prediction = analysis.get("prediction")
    if prediction not in {"CONSISTENT", "INCONSISTENT"}:
        raise ValueError(f"Invalid binary prediction: {prediction!r}")
    count_keys = ("inconsistency_tp", "inconsistency_fp", "inconsistency_fn")
    counts = [analysis.get(key) for key in count_keys]
    if not all(isinstance(value, int) and not isinstance(value, bool) and value >= 0 for value in counts):
        raise TypeError("GPT analysis must return non-negative integer case counts")
    return prediction, {
        "TP": counts[0],
        "FP": counts[1],
        "FN": counts[2],
    }


In [3]:
metadata_records = load_metadata()
metadata_file_names = [item["Image Name"] for item in metadata_records]
core_results = load_results(CORE_RESULTS_PATH)
raw_results = load_results(RAW_RESULTS_PATH)
core_by_file = {item["file_name"]: item for item in core_results}
raw_by_file = {item["file_name"]: item for item in raw_results}

checkpoint_names = metadata_file_names[:x]
if len(core_by_file) < x or len(raw_by_file) < x:
    raise RuntimeError(
        f"Checkpoint x={x} requires at least x records in both result files; "
        f"found core={len(core_by_file)}, raw={len(raw_by_file)}."
    )

core_by_file = {
    name: core_by_file[name]
    for name in checkpoint_names
    if name in core_by_file
}
raw_by_file = {
    name: raw_by_file[name]
    for name in checkpoint_names
    if name in raw_by_file
}
if set(core_by_file) != set(checkpoint_names) or set(raw_by_file) != set(checkpoint_names):
    raise RuntimeError(
        "The first x metadata records are not present in both result files. "
        "Set x to a valid completed-record checkpoint."
    )
for name, item in core_by_file.items():
    counts = item.get("inconsistency_cases")
    if not isinstance(counts, dict) or set(counts) != {"TP", "FP", "FN"}:
        raise RuntimeError(
            f"Existing baseline result for {name} uses the old schema. "
            "Set x=0 to regenerate case-level TP/FP/FN results."
        )

save_results(CORE_RESULTS_PATH, list(core_by_file.values()))
save_results(RAW_RESULTS_PATH, list(raw_by_file.values()))
existing_files = set(checkpoint_names)

print(f"Checkpoint records kept: {x}")
print(f"Next sample: {metadata_file_names[x] if x < len(metadata_file_names) else 'none; all samples complete'}")

Checkpoint records kept: 0
Next sample: s1r1_consistent.jpg


In [ ]:
failed_files = []
processed_this_run = 0

for metadata_index, record in enumerate(
    metadata_records[x:x + n_samples],
    start=x + 1,
):
    file_name = record["Image Name"]
    if file_name in existing_files:
        continue

    print(f"[{metadata_index}/108] Running baseline {file_name}")
    start_time = time.perf_counter()
    try:
        image_path = find_image(file_name)
        prompt = f"""
            You audit semantic consistency between an AR application rule and the virtual
            content shown in an AR image. Identify every relevant entity and evaluate each
            one individually. Do not rely only on the full-image impression.

            Carefully distinguish physical markings, printed borders, and object edges
            from virtual AR overlays. Use scene geometry, occlusion, alignment, and the
            overlay's visual style as evidence. Contrast or edge sharpness may support the
            judgment, but must not be used alone to classify content as virtual.

            As a general priority, when the rule mainly concerns an AR cue's color, shape,
            size, style, or presence, consider grounding the virtual cue first. If the cue
            is unclear or its target relationship is difficult to establish, ground one or
            more relevant physical objects as a complementary route. This is a preference,
            not a fixed order; choose the route that is most informative for the image.

            This experiment requires a binary decision. Do not output
            INSUFFICIENT_EVIDENCE as an entity or overall classification. State the confidence level (high, medium,
            or low), explain the evidence supporting the choice, and explicitly mention
            the limitation or ambiguity that reduced confidence.

            For each relevant entity classify it as CONSISTENT or INCONSISTENT. The final
            answer must list the entities evaluated, the binary conclusion for each,
            supporting evidence, confidence levels, all violations, and an overall binary
            conclusion.

            Do not use visual tools because this is the GPT baseline.

            Application: {record["Application"]}
            Rule: {record["Rule"]}
            Reference inconsistency type is hidden from the prediction task.
        """
        response = client.responses.create(
            model=MODEL,
            reasoning={"effort": THINKING},
            input=[
                {"role": "user", "content": [
                    {"type": "input_text", "text": prompt},
                    {"type": "input_image", "image_url": image_data_url(image_path)},
                ]},
            ],
        )
        agent_output = response.output_text
        latency_seconds = round(time.perf_counter() - start_time, 3)
        prediction, inconsistency_cases = analyze_baseline_output(
            record,
            agent_output,
        )
        binary_annotation = int(record["Binary Annotation"]) == 1
        binary_match = (prediction == "INCONSISTENT") == binary_annotation

        core_by_file[file_name] = {
            "file_name": file_name,
            "binary_match": binary_match,
            "inconsistency_cases": inconsistency_cases,
            "tool_counts": {},
            "latency_seconds": latency_seconds,
        }
        raw_by_file[file_name] = {
            "file_name": file_name,
            "agent_results": agent_output,
        }
        save_results(CORE_RESULTS_PATH, list(core_by_file.values()))
        save_results(RAW_RESULTS_PATH, list(raw_by_file.values()))
        existing_files.add(file_name)
        processed_this_run += 1
        print(
            f"Finished {file_name}: binary={binary_match}, "
            f"inconsistency_cases={inconsistency_cases}, tool_calls={{}}, "
            f"latency={latency_seconds:.3f}s"
        )
    except Exception as error:
        failed_files.append({"file_name": file_name, "error": repr(error)})
        print(f"FAILED {file_name}: {error}")

print(f"Completed this run: {processed_this_run}")
print(f"Failed files in this run: {len(failed_files)}")

[1/108] Running baseline s1r1_consistent.jpg


<>:64: SyntaxWarning: invalid escape sequence '\_'
<>:64: SyntaxWarning: invalid escape sequence '\_'
/tmp/ipykernel_851345/2055539183.py:64: SyntaxWarning: invalid escape sequence '\_'
  binary_annotation = int(record["Binary Annotation"]) == 1


KeyboardInterrupt: 

In [ ]:
final_core_results = load_results(CORE_RESULTS_PATH)
final_raw_results = load_results(RAW_RESULTS_PATH)
print("Core records:", len(final_core_results))
print("Raw records:", len(final_raw_results))
print("Binary matches:", sum(item["binary_match"] for item in final_core_results))
print("Case TP:", sum(item["inconsistency_cases"]["TP"] for item in final_core_results))
print("Case FP:", sum(item["inconsistency_cases"]["FP"] for item in final_core_results))
print("Case FN:", sum(item["inconsistency_cases"]["FN"] for item in final_core_results))
print("Failed files in this run:", failed_files)